In [ ]:
# Cell 1: Clone the repository
import subprocess, sys
result = subprocess.run(
    ['git', 'clone', 'https://github.com/EugenTheMachine/instanseg.git', 'repo'],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)

In [ ]:
# Cell 2: Reinstall PyTorch to CUDA 11.8 version compatible with Tesla P100 (sm_60) & install dependencies
import subprocess, sys

print('=== Reinstalling PyTorch 2.3.1 (cu118) for P100 sm_60 compatibility ===')
torch_reinstall = subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall',
    'torch==2.3.1+cu118', 'torchvision==0.18.1+cu118',
    '--extra-index-url', 'https://download.pytorch.org/whl/cu118'
], capture_output=True, text=True)
print('PyTorch Reinstall Status:', 'OK' if torch_reinstall.returncode == 0 else 'FAILED')
if torch_reinstall.returncode != 0:
    print(torch_reinstall.stderr[-600:])

def pip_install(pkgs, no_deps=False, force_reinstall=False):
    for pkg in pkgs:
        cmd = [sys.executable, '-m', 'pip', 'install', '-q']
        if no_deps:
            cmd.append('--no-deps')
        if force_reinstall:
            cmd.append('--force-reinstall')
        cmd.append(pkg)
        r = subprocess.run(cmd, capture_output=True, text=True)
        status = 'OK' if r.returncode == 0 else 'FAILED'
        print(f'{status}: {pkg}')
        if r.returncode != 0:
            print(r.stderr[-600:])

safe_packages = [
    'einops>=0.7.0',
    'fastremap>=1.14.0',
    'tifffile>=2023.12.9',
    'scikit-image>=0.22.0',
    'albumentations',
    'tqdm>=4.66.1',
    'pyyaml',
    'pandas',
    'opencv-python-headless',
    'stardist',
    'colorcet',
    'geojson',
    'palettable',
    'rasterio',
    'scipy',
    'seaborn',
    'psutil',
]
print('=== Installing safe packages ===')
pip_install(safe_packages)

torch_dep_packages = [
    'ultralytics',
    'monai',
    'torchstain',
]
print('=== Installing torch-dependent packages (--no-deps) ===')
pip_install(torch_dep_packages, no_deps=True)

# Force-reinstall Pillow to a version that has PIL._typing._Ink (>=10.2.0).
# The torch/torchvision reinstall above can downgrade Pillow to an older version
# that is missing this symbol, causing an ImportError on import of torchvision.
print('=== Force-reinstalling Pillow to ensure PIL._typing._Ink is available ===')
pip_install(['Pillow==10.4.0'], force_reinstall=True)

In [ ]:
# Cell 3: Verify torch has P100 CUDA support
import torch
print(f'torch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'CUDA version: {torch.version.cuda}')
else:
    raise RuntimeError('CUDA not available! torch was likely overwritten during install.')

In [ ]:
# Cell 4: Add repo to path and verify import
import sys
sys.path.insert(0, 'repo')

from instanseg import InstanSegModel
print('InstanSegModel imported successfully')

In [ ]:
# Cell 5: Locate and verify the mounted Kaggle dataset
import os
from pathlib import Path

KAGGLE_INPUT_DIR = Path('/kaggle/input')
PREFERRED_DATA_DIR = KAGGLE_INPUT_DIR / 'overfit-small-dataset'


def has_image_mask_layout(path):
    image_dirs = list(path.rglob('images'))
    mask_dirs = list(path.rglob('masks'))
    return any(image_dir.is_dir() and mask_dir.is_dir() for image_dir in image_dirs for mask_dir in mask_dirs)


if PREFERRED_DATA_DIR.exists() and has_image_mask_layout(PREFERRED_DATA_DIR):
    DATA_DIR = str(PREFERRED_DATA_DIR)
else:
    dataset_roots = [
        path for path in KAGGLE_INPUT_DIR.rglob('*')
        if path.is_dir() and has_image_mask_layout(path)
    ]
    if dataset_roots:
        DATA_DIR = str(min(dataset_roots, key=lambda path: len(path.parts)))
    else:
        mounted_entries = sorted(str(path) for path in KAGGLE_INPUT_DIR.glob('*'))
        raise FileNotFoundError(
            'No mounted dataset containing images/ and masks/ directories was found. '
            f'Expected {PREFERRED_DATA_DIR} or another attached dataset under {KAGGLE_INPUT_DIR}. '
            f'Mounted entries: {mounted_entries}'
        )

CONFIG_PATH = 'repo/config.yaml'
print(f'Data dir: {os.path.abspath(DATA_DIR)}')
data = Path(DATA_DIR)
train_imgs = list((data / 'train' / 'images').glob('*'))
train_masks = list((data / 'train' / 'masks').glob('*'))
if len(train_imgs) == 0:
    train_imgs = list((data / 'images').glob('*'))
    train_masks = list((data / 'masks').glob('*'))
print(f'Train images: {len(train_imgs)}, Train masks: {len(train_masks)}')

In [ ]:
# Cell 6: Train InstanSeg_UNet with center-seed embeddings
import torch
print(f'Training on: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

center_seed_model = InstanSegModel(
    config_path=CONFIG_PATH,
    model_name='InstanSeg_UNet',
    embedding_mode='center-seed',
    experiment_name='instanseg_unet_center_seed',
)
center_seed_metrics = center_seed_model.train(
    epochs=50,
    imgsz=512,
    batch_size=4,
    train_data_ratio=1.0,
    data_dir=DATA_DIR,
    num_workers=8,
    embedding_mode='center-seed',
    patience=5
)
print('Center-seed training complete!')
print('Metrics:', center_seed_metrics)

In [ ]:
# Cell 7: Train InstanSeg_UNet with border-cluster embeddings
import torch
print(f'Training on: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

border_cluster_model = InstanSegModel(
    config_path=CONFIG_PATH,
    model_name='InstanSeg_UNet',
    embedding_mode='border-cluster',
    experiment_name='instanseg_unet_border_cluster',
)
border_cluster_metrics = border_cluster_model.train(
    epochs=50,
    imgsz=512,
    batch_size=4,
    train_data_ratio=1.0,
    data_dir=DATA_DIR,
    num_workers=8,
    embedding_mode='border-cluster',
    patience=5
)
print('Border-cluster training complete!')
print('Metrics:', border_cluster_metrics)